# Download the ORD datasets reviewed in the HTML

Run the cells in order. The destination is editable in **Settings**.

This notebook downloads the **13 exports across 7 papers** in the final ORD table of `Reaction_Dataset_Comparison.html` (10 October 2026). Filenames follow `FirstAuthor_ORD-folder-code_Reaction_Subset.parquet`; compound author surnames are kept. No HTML file is needed to run it.

Files come directly from GitHub's **Git LFS media endpoint**, rather than the small pointer files served by `raw.githubusercontent.com`. The repository revision is pinned to the snapshot cited in the HTML. Dataset IDs and row counts are checked before a download is accepted; existing valid files are skipped.

Scope follows the HTML: HiTEA and Ahneman are ORD exports with different counts from the author releases; Newman-Stonebraker covers the Ni Suzuki subset; Gesmundo's reductive-amination export combines two sheets. `Reizman_20` retains the HTML's warning about the exported electrophile discrepancy, without assigning an unverified case label.

Sources: [ORD GitHub repository](https://github.com/open-reaction-database/ord-data) · [ORD browse page](https://open-reaction-database.org/browse). A small `download_manifest.csv` records the renamed filenames, full IDs, URLs, counts, and the HTML's scope notes.


In [ ]:
# # Run once per notebook environment.
# %pip install -q requests pyarrow

Note: you may need to restart the kernel to use updated packages.


c:\venvs\GP_hub\Scripts\python.exe: No module named pip


In [2]:
from pathlib import Path
import csv
import requests
import pyarrow.parquet as pq
from requests.adapters import HTTPAdapter

# SETTINGS: change this folder if needed.
DESTINATION = Path(r"C:\Users\koand\OneDrive\Documents\GitHub\DOPE-MURI\ORD_files\ORD_datasets_parquet")
OVERWRITE = False
GITHUB_REF = "83f971f586f6ad18f358ae4ae99d045e94ed2066"  # Snapshot cited in the HTML


In [3]:
# Filename, full ORD dataset ID, expected record count (from the HTML).
DATASETS = [
    ('Reizman_31_Suzuki_3-bromoquinoline_isoxazole-Bpin.parquet', 'ord_dataset-31989f1b2b9d4885b1dd2d9982da4517', 96),
    ('Reizman_bc_Suzuki_3-chloropyridine_isoxazole-Bpin.parquet', 'ord_dataset-bc349c19b4384756a2aee8aa525b6c2a', 96),
    ('Reizman_20_Suzuki_benzofuran-boronic-acid.parquet', 'ord_dataset-2038a3c967db4a32a8fbce288437e929', 96),
    ('Reizman_2b_Suzuki_2-chloropyridine_N-Boc-pyrrole-boronic-acid.parquet', 'ord_dataset-2be30f5d8dcd471aa6ad410bdee05902', 97),
    ('King-Smith_d9_mixed-reactions_HiTEA.parquet', 'ord_dataset-d92976309c3a48a3a64a4cf5e7048086', 39347),
    ('Buitrago-Santanilla_cb_Buchwald-Hartwig_model-screen_triplicates.parquet', 'ord_dataset-cbcc4048add7468e850b6ec42549c70d', 288),
    ('Buitrago-Santanilla_7d_cross-coupling_Experiment-2.parquet', 'ord_dataset-7d8f5fd922d4497d91cb81489b052746', 1536),
    ('Ahneman_46_Buchwald-Hartwig_fragment-additives.parquet', 'ord_dataset-46ff9a32d9e04016b9380b1b1ef949c3', 4312),
    ('Perera_68_Suzuki_nanoscale-flow.parquet', 'ord_dataset-68cb8b4b2b384e3d85b5b1efae58b203', 5760),
    ('Gesmundo_3b_Suzuki_12x10-informer-library.parquet', 'ord_dataset-3b8a2ef300e145468579027f206a3ac8', 1440),
    ('Gesmundo_2b_reductive-amination_optimization-and-library.parquet', 'ord_dataset-2be11f57f3304e678ea8469baf6dd1bc', 1152),
    ('Gesmundo_17_N-alkylation_Boc-deprotection.parquet', 'ord_dataset-172039a759a440219a68af62d203b79b', 104),
    ('Newman-Stonebraker_3b_Suzuki_Ni-ligand-screens.parquet', 'ord_dataset-3b5db90e337942ea886b8f5bc5e3aa72', 450),
]

# Scope notes copied from the corresponding HTML table rows.
SCOPE_NOTES = {
    "ord_dataset-31989f1b2b9d4885b1dd2d9982da4517": "3-bromoquinoline + isoxazole Bpin; 96 ORD records. These are exported records, not the 92 optimization experiments cited in the paper figure.",
    "ord_dataset-bc349c19b4384756a2aee8aa525b6c2a": "3-chloropyridine + isoxazole Bpin; 96 ORD records.",
    "ord_dataset-2038a3c967db4a32a8fbce288437e929": "Benzofuran-2-boronic acid screen; ORD encodes 2-chloropyridine, whereas paper case III uses 3-chloropyridine. Preserve this discrepancy when importing.",
    "ord_dataset-2be30f5d8dcd471aa6ad410bdee05902": "2-chloropyridine + N-Boc-pyrrole-2-boronic acid; 97 ORD records.",
    "ord_dataset-d92976309c3a48a3a64a4cf5e7048086": "39,347 ORD records versus 39,546 rows in the author raw release. Solvent/reagent mixtures have additional component and SMILES annotation; counts are not interchangeable.",
    "ord_dataset-cbcc4048add7468e850b6ec42549c70d": "Model C\u2013N screen: 3-bromopyridine + 4-phenylpiperidine; 96 unique input/condition settings \u00d7 triplicate = 288 records. Matched through the reaction publication URL (Science 347, 49\u201353), because the dataset description omits the DOI.",
    "ord_dataset-7d8f5fd922d4497d91cb81489b052746": "Experiment 2: 1,536-record cross-coupling screen. This export is a subset of the paper, not all experimental blocks.",
    "ord_dataset-46ff9a32d9e04016b9380b1b1ef949c3": "4,312 ORD records. Distinct release scope from the 4,599-row author identity table and 3,960-row descriptor matrix; do not substitute these counts or add the overlapping releases.",
    "ord_dataset-68cb8b4b2b384e3d85b5b1efae58b203": "Principal 5,760-reaction nanoscale flow Suzuki screen; corresponds to the S1 workbook scope. The separate 576-setting bromo-oxindole screen is not an additional block here.",
    "ord_dataset-3b8a2ef300e145468579027f206a3ac8": "Suzuki informer library: 12 cores \u00d7 10 boron partners \u00d7 12 recipes = 1,440 records; corresponds to suzuki_informer.",
    "ord_dataset-2be11f57f3304e678ea8469baf6dd1bc": "Both reductive-amination sheets combined: 768 optimization + 384 library-synthesis records = 1,152. The experiment set identifier distinguishes the blocks.",
    "ord_dataset-172039a759a440219a68af62d203b79b": "N-alkylation / in situ Boc deprotection: 96 reaction records + 8 deprotection controls = 104; corresponds to alkylation_deprotection.",
    "ord_dataset-3b5db90e337942ea886b8f5bc5e3aa72": "Ni Suzuki ligand-screen subset: 450 records. Dataset description cites the Science paper; reaction provenance cites its ChemRxiv preprint, 10.26434/chemrxiv.14388557.v1. This file does not include every paper dataset."
}


In [4]:
DESTINATION.mkdir(parents=True, exist_ok=True)
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=3))

def check_parquet(path, dataset_id, expected_rows):
    metadata = pq.read_metadata(path)
    actual_id = (metadata.metadata or {}).get(b"ord.dataset_id", b"").decode()
    if actual_id != dataset_id or metadata.num_rows != expected_rows:
        raise ValueError(f"Unexpected dataset/row count in {path.name}: {actual_id}, {metadata.num_rows}")

def download_one(filename, dataset_id, expected_rows):
    code = dataset_id.removeprefix("ord_dataset-")[:2]
    repo_path = f"data/{code}/{dataset_id}.parquet"
    url = f"https://media.githubusercontent.com/media/open-reaction-database/ord-data/{GITHUB_REF}/{repo_path}"
    target = DESTINATION / filename
    if target.exists() and not OVERWRITE:
        check_parquet(target, dataset_id, expected_rows)
        status = "already valid"
    else:
        temporary = target.with_suffix(".parquet.part")
        try:
            with session.get(url, stream=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                with temporary.open("wb") as stream:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            stream.write(chunk)
            check_parquet(temporary, dataset_id, expected_rows)
            temporary.replace(target)
            status = "downloaded"
        finally:
            temporary.unlink(missing_ok=True)
    print(f"{status:13} | {expected_rows:>6,} records | {filename}")
    return dict(filename=filename, dataset_id=dataset_id, records=expected_rows,
                status=status, github_url=f"https://github.com/open-reaction-database/ord-data/blob/{GITHUB_REF}/{repo_path}",
                download_url=url, scope_note=SCOPE_NOTES[dataset_id])

manifest, failures = [], []
for item in DATASETS:
    try:
        manifest.append(download_one(*item))
    except Exception as error:
        failures.append(item[0])
        print(f"FAILED | {item[0]} | {error}")
if manifest:
    with (DESTINATION / "download_manifest.csv").open("w", newline="", encoding="utf-8-sig") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(manifest[0]))
        writer.writeheader()
        writer.writerows(manifest)
session.close()
if failures:
    raise RuntimeError(f"{len(failures)} download(s) failed. Fix the reported issue and rerun; valid files will be skipped.")
print(f"\nVerified all {len(manifest)} Parquet files in {DESTINATION}")


downloaded    |     96 records | Reizman_31_Suzuki_3-bromoquinoline_isoxazole-Bpin.parquet
downloaded    |     96 records | Reizman_bc_Suzuki_3-chloropyridine_isoxazole-Bpin.parquet
downloaded    |     96 records | Reizman_20_Suzuki_benzofuran-boronic-acid.parquet
downloaded    |     97 records | Reizman_2b_Suzuki_2-chloropyridine_N-Boc-pyrrole-boronic-acid.parquet
downloaded    | 39,347 records | King-Smith_d9_mixed-reactions_HiTEA.parquet
downloaded    |    288 records | Buitrago-Santanilla_cb_Buchwald-Hartwig_model-screen_triplicates.parquet
downloaded    |  1,536 records | Buitrago-Santanilla_7d_cross-coupling_Experiment-2.parquet
downloaded    |  4,312 records | Ahneman_46_Buchwald-Hartwig_fragment-additives.parquet
downloaded    |  5,760 records | Perera_68_Suzuki_nanoscale-flow.parquet
downloaded    |  1,440 records | Gesmundo_3b_Suzuki_12x10-informer-library.parquet
downloaded    |  1,152 records | Gesmundo_2b_reductive-amination_optimization-and-library.parquet
downloaded    |